In [3]:
import os
import sys
import json
import re
import random
from pathlib import Path
from typing import Any, Dict, List, Set, Tuple

import numpy as np


# ============================================================
# Paths you may need to change
# ============================================================

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "evaluation/dataset" / "hard_triplets_tfidf_dbpedia.json"
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints/dbpedia_model_semantic-20.pt"

EXAMPLE_INDEX = 0
TOP_K = 3
BATCH_SIZE = 32
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

from src.embedder import SPARQLEmbedder


# ============================================================
# Basic query extraction helpers
# ============================================================

def get_query_text(x: Any) -> str:
    if x is None:
        return ""

    if isinstance(x, dict):
        return str(
            x.get("query")
            or x.get("sparql")
            or x.get("anchor_query")
            or ""
        ).strip()

    return str(x).strip()


def get_anchor_query(item: Dict[str, Any]) -> str:
    return get_query_text(item.get("anchor_query") or item.get("base_query"))


def iter_labeled_queries(item: Dict[str, Any], positives_only: bool = False) -> List[str]:
    out = []

    for p in item.get("positive_queries") or []:
        q = get_query_text(p)
        if q:
            out.append(q)

    if not positives_only:
        for n in item.get("negative_queries") or []:
            q = get_query_text(n)
            if q:
                out.append(q)

    return out


def build_global_corpus(raw_data: List[Dict[str, Any]]) -> Tuple[List[str], Set[str]]:
    unique_queries = []
    seen_q = set()

    def add_unique(q: str):
        if q and q not in seen_q:
            seen_q.add(q)
            unique_queries.append(q)

    for item in raw_data:
        add_unique(get_anchor_query(item))

        for q in iter_labeled_queries(item, positives_only=False):
            add_unique(q)

    return unique_queries, seen_q


def build_eval_items(
    raw_data: List[Dict[str, Any]],
    global_corpus: List[str],
    seen_q: Set[str],
):
    eval_items = []

    for item in raw_data:
        anchor_q = get_anchor_query(item)

        if not anchor_q or anchor_q not in seen_q:
            continue

        positives = set()

        for p in item.get("positive_queries") or []:
            q = get_query_text(p)
            if q and q in seen_q and q != anchor_q:
                positives.add(q)

        if positives:
            eval_items.append({
                "anchor_query": anchor_q,
                "positives": positives,
                "raw_item": item,
            })

    return eval_items


# ============================================================
# Perturbation helpers
# ============================================================

PREFIX_RE = re.compile(
    r"^\s*(?:PREFIX\s+[^\s:]*:?\s*<[^>]*>|BASE\s+<[^>]*>)\s*",
    re.IGNORECASE | re.DOTALL,
)

COUNT_RE = re.compile(r"\bCOUNT\s*\(", re.IGNORECASE)

VAR_TOKEN_RE = re.compile(
    r"(?<![A-Za-z0-9_])([?$][A-Za-z_][A-Za-z0-9_]*)(?![A-Za-z0-9_])"
)


def normalize_query(q: str) -> str:
    return " ".join(str(q or "").strip().split())


def remove_sparql_comments(q: str) -> str:
    q = str(q or "")

    out = []
    in_string = False
    string_char = None
    in_iri = False

    i = 0
    while i < len(q):
        ch = q[i]

        if in_string:
            out.append(ch)

            if ch == "\\":
                if i + 1 < len(q):
                    out.append(q[i + 1])
                    i += 2
                    continue

            if ch == string_char:
                in_string = False
                string_char = None

            i += 1
            continue

        if in_iri:
            out.append(ch)

            if ch == ">":
                in_iri = False

            i += 1
            continue

        if ch in ["'", '"']:
            in_string = True
            string_char = ch
            out.append(ch)
            i += 1
            continue

        if ch == "<":
            in_iri = True
            out.append(ch)
            i += 1
            continue

        if ch == "#":
            while i < len(q) and q[i] not in ["\n", "\r"]:
                i += 1
            out.append(" ")
            continue

        out.append(ch)
        i += 1

    return "".join(out)


def strip_prefixes_and_comments(q: str) -> str:
    q = remove_sparql_comments(q).strip()

    while True:
        m = PREFIX_RE.match(q)
        if not m:
            break
        q = q[m.end():].strip()

    return q


def is_select_query(q: str) -> bool:
    body = strip_prefixes_and_comments(q)
    return bool(re.match(r"(?is)^SELECT\b", body))


def is_count_query(q: str) -> bool:
    return bool(COUNT_RE.search(str(q or "")))


def has_select_star(q: str) -> bool:
    body = strip_prefixes_and_comments(q)
    return bool(re.search(r"(?is)^SELECT\s+(?:DISTINCT\s+|REDUCED\s+)?\*", body))


def find_where_body_bounds(query: str):
    m = re.search(r"\bWHERE\s*\{", query, re.IGNORECASE)

    if m:
        open_brace = query.find("{", m.start())
    else:
        open_brace = query.find("{")

    if open_brace < 0:
        return None

    depth = 0
    in_string = False
    string_char = None
    in_iri = False

    i = open_brace
    while i < len(query):
        ch = query[i]

        if in_string:
            if ch == "\\":
                i += 2
                continue

            if ch == string_char:
                in_string = False
                string_char = None

            i += 1
            continue

        if in_iri:
            if ch == ">":
                in_iri = False

            i += 1
            continue

        if ch in ["'", '"']:
            in_string = True
            string_char = ch
            i += 1
            continue

        if ch == "<":
            in_iri = True
            i += 1
            continue

        if ch == "{":
            depth += 1

        elif ch == "}":
            depth -= 1

            if depth == 0:
                return open_brace + 1, i

        i += 1

    return None


def collect_variables_outside_strings_and_iris(query: str) -> List[str]:
    seen = []

    in_string = False
    string_char = None
    in_iri = False

    i = 0
    while i < len(query):
        ch = query[i]

        if in_string:
            if ch == "\\":
                i += 2
                continue

            if ch == string_char:
                in_string = False
                string_char = None

            i += 1
            continue

        if in_iri:
            if ch == ">":
                in_iri = False

            i += 1
            continue

        if ch in ["'", '"']:
            in_string = True
            string_char = ch
            i += 1
            continue

        if ch == "<":
            in_iri = True
            i += 1
            continue

        m = VAR_TOKEN_RE.match(query, i)
        if m:
            var_token = m.group(1)
            if var_token not in seen:
                seen.append(var_token)
            i = m.end()
            continue

        i += 1

    return seen


def replace_variables_outside_strings_and_iris(query: str, mapping: Dict[str, str]) -> str:
    out = []

    in_string = False
    string_char = None
    in_iri = False

    i = 0
    while i < len(query):
        ch = query[i]

        if in_string:
            out.append(ch)

            if ch == "\\":
                if i + 1 < len(query):
                    out.append(query[i + 1])
                    i += 2
                    continue

            if ch == string_char:
                in_string = False
                string_char = None

            i += 1
            continue

        if in_iri:
            out.append(ch)

            if ch == ">":
                in_iri = False

            i += 1
            continue

        if ch in ["'", '"']:
            in_string = True
            string_char = ch
            out.append(ch)
            i += 1
            continue

        if ch == "<":
            in_iri = True
            out.append(ch)
            i += 1
            continue

        m = VAR_TOKEN_RE.match(query, i)
        if m:
            old_var = m.group(1)
            out.append(mapping.get(old_var, old_var))
            i = m.end()
            continue

        out.append(ch)
        i += 1

    return "".join(out)


def rename_variables(query: str) -> str:
    seen = collect_variables_outside_strings_and_iris(query)

    if not seen:
        return query

    readable_pool = [
        "entity",
        "subject",
        "object",
        "resource",
        "result",
        "value",
        "label",
        "name",
        "type",
        "category",
        "classType",
        "property",
        "relation",
        "target",
        "source",
        "item",
        "node",
        "concept",
        "answer",
        "candidate",
    ]

    fallback_pool = [
        "alpha",
        "beta",
        "gamma",
        "delta",
        "epsilon",
        "zeta",
        "theta",
        "lambda",
        "omega",
    ]

    existing_vars = set(seen)
    used_new_vars = set()
    mapping = {}

    def next_available_name(base_name: str) -> str:
        candidate = f"?{base_name}"

        if candidate not in existing_vars and candidate not in used_new_vars:
            return candidate

        for suffix in fallback_pool:
            candidate2 = f"?{base_name}{suffix.capitalize()}"

            if candidate2 not in existing_vars and candidate2 not in used_new_vars:
                return candidate2

        candidate3 = f"?{base_name}Alt"

        while candidate3 in existing_vars or candidate3 in used_new_vars:
            candidate3 += "Alt"

        return candidate3

    for i, old_var in enumerate(seen):
        if i < len(readable_pool):
            new_var = next_available_name(readable_pool[i])
        else:
            new_var = next_available_name("entity")

        mapping[old_var] = new_var
        used_new_vars.add(new_var)

    return replace_variables_outside_strings_and_iris(query, mapping)


def split_top_level_dot_patterns(where_body: str) -> List[str]:
    parts = []
    buf = []

    depth = 0
    in_string = False
    string_char = None
    in_iri = False

    i = 0
    while i < len(where_body):
        ch = where_body[i]

        if in_string:
            buf.append(ch)

            if ch == "\\":
                if i + 1 < len(where_body):
                    buf.append(where_body[i + 1])
                    i += 2
                    continue

            if ch == string_char:
                in_string = False
                string_char = None

            i += 1
            continue

        if in_iri:
            buf.append(ch)

            if ch == ">":
                in_iri = False

            i += 1
            continue

        if ch in ["'", '"']:
            in_string = True
            string_char = ch
            buf.append(ch)
            i += 1
            continue

        if ch == "<":
            in_iri = True
            buf.append(ch)
            i += 1
            continue

        if ch == "{":
            depth += 1
            buf.append(ch)
            i += 1
            continue

        if ch == "}":
            depth -= 1
            buf.append(ch)
            i += 1
            continue

        if ch == "." and depth == 0:
            part = "".join(buf).strip()

            if part:
                parts.append(part)

            buf = []
            i += 1
            continue

        buf.append(ch)
        i += 1

    tail = "".join(buf).strip()

    if tail:
        parts.append(tail)

    return parts


def is_safe_reorder_pattern(pattern: str) -> bool:
    upper = pattern.strip().upper()

    unsafe_keywords = [
        "BIND",
        "OPTIONAL",
        "UNION",
        "VALUES",
        "SERVICE",
        "GRAPH",
        "MINUS",
        "SELECT",
        "ASK",
        "CONSTRUCT",
        "DESCRIBE",
    ]

    for keyword in unsafe_keywords:
        if re.search(rf"\b{keyword}\b", upper):
            return False

    if re.search(r"\bFILTER\s+EXISTS\b", upper):
        return False

    if re.search(r"\bFILTER\s+NOT\s+EXISTS\b", upper):
        return False

    return True


def reorder_triples(query: str) -> str:
    bounds = find_where_body_bounds(query)

    if not bounds:
        return query

    start, end = bounds
    body = query[start:end]

    patterns = split_top_level_dot_patterns(body)

    if len(patterns) <= 1:
        return query

    if not all(is_safe_reorder_pattern(p) for p in patterns):
        return query

    shuffled = patterns[:]
    random.shuffle(shuffled)

    if shuffled == patterns:
        shuffled = shuffled[::-1]

    new_body = "\n  " + " .\n  ".join(shuffled) + " .\n"

    return query[:start] + new_body + query[end:]


def insert_dummy_optional(query: str) -> str:
    if has_select_star(query):
        return query

    bounds = find_where_body_bounds(query)

    if not bounds:
        return query

    _, end = bounds

    noop_optional = """
  OPTIONAL {
    FILTER(false)
  }
"""

    return query[:end] + noop_optional + query[end:]


def inject_limit_offset(query: str) -> str:
    if re.search(r"\bOFFSET\b", query, re.IGNORECASE):
        return query

    stripped = query.rstrip().rstrip(";").rstrip()
    return stripped + "\nOFFSET 0"


def combined_perturbation(query: str) -> str:
    q = rename_variables(query)
    q = reorder_triples(q)
    q = insert_dummy_optional(q)
    q = inject_limit_offset(q)
    return q


def make_perturbations(query: str) -> List[Dict[str, Any]]:
    if not is_select_query(query):
        return []

    if is_count_query(query):
        return []

    perturbation_fns = [
        ("variable_renaming", rename_variables),
        ("triple_reordering", reorder_triples),
        ("optional_insertion", insert_dummy_optional),
        ("limit_offset_injection", inject_limit_offset),
        ("combined", combined_perturbation),
    ]

    valid = []

    for name, fn in perturbation_fns:
        perturbed = fn(query)

        if normalize_query(perturbed) == normalize_query(query):
            continue

        valid.append({
            "type": name,
            "query": perturbed,
            "expected_answer_jaccard": 1.0,
            "validated_with_endpoint": False,
        })

    return valid


# ============================================================
# Load one example and retrieve top 3 using SPARQLCE
# ============================================================

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

global_corpus, seen_q = build_global_corpus(raw_data)
eval_items = build_eval_items(raw_data, global_corpus, seen_q)

print(f"Raw items: {len(raw_data)}")
print(f"Global corpus size: {len(global_corpus)}")
print(f"Eval items with positives: {len(eval_items)}")

example = eval_items[EXAMPLE_INDEX]
anchor_query = example["anchor_query"]
positives = example["positives"]

print("\nSelected example index:", EXAMPLE_INDEX)
print("Number of annotated positives:", len(positives))

model = SPARQLEmbedder(str(CHECKPOINT_PATH))

emb_map = {}

for i in range(0, len(global_corpus), BATCH_SIZE):
    batch = global_corpus[i:i + BATCH_SIZE]
    embs = model.encode(batch)

    if not isinstance(embs, np.ndarray):
        embs = np.array(embs)

    embs = embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-12)

    for q, e in zip(batch, embs):
        emb_map[q] = e

    print(f"Encoded {i + len(batch)}/{len(global_corpus)}")

corpus_embs = np.vstack([emb_map[q] for q in global_corpus])
anchor_emb = emb_map[anchor_query]

scores = corpus_embs @ anchor_emb

anchor_idx = global_corpus.index(anchor_query)
scores[anchor_idx] = np.finfo(scores.dtype).min

order = np.argsort(-scores)

top3 = []

for rank, idx in enumerate(order[:TOP_K], start=1):
    q = global_corpus[idx]
    top3.append({
        "rank": rank,
        "similarity_score": float(scores[idx]),
        "is_annotated_positive": q in positives,
        "query": q,
        "valid_perturbations": make_perturbations(q),
    })

example_output = {
    "example_index": EXAMPLE_INDEX,
    "anchor": {
        "query": anchor_query,
        "valid_perturbations": make_perturbations(anchor_query),
    },
    "retrieved_top3": top3,
    "annotated_positive_count": len(positives),
    "retrieved_positive_count": sum(1 for x in top3 if x["is_annotated_positive"]),
}

print("\n" + "=" * 100)
print("ONE RETRIEVAL-STYLE ROBUSTNESS EXAMPLE")
print("=" * 100)

print(json.dumps(example_output, indent=2, ensure_ascii=False))

OUTPUT_PATH = PROJECT_ROOT / "dataset" / f"one_rq2_retrieval_top3_example_{EXAMPLE_INDEX}.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(example_output, f, indent=2, ensure_ascii=False)

print("\nSaved example JSON to:")
print(OUTPUT_PATH)

Raw items: 1428
Global corpus size: 1830
Eval items with positives: 1428

Selected example index: 0
Number of annotated positives: 31


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1396.36it/s, Materializing param=pooler.dense.weight]                               


Loading model from /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/checkpoints/dbpedia_model_semantic-20.pt...
Encoded 32/1830
Encoded 64/1830
Encoded 96/1830
Encoded 128/1830
Encoded 160/1830
Encoded 192/1830
Encoded 224/1830
Encoded 256/1830
Encoded 288/1830
Encoded 320/1830
Encoded 352/1830
Encoded 384/1830
Encoded 416/1830
Encoded 448/1830
Encoded 480/1830
Encoded 512/1830
Encoded 544/1830
Encoded 576/1830
Encoded 608/1830
Encoded 640/1830
Encoded 672/1830
Encoded 704/1830
Encoded 736/1830
Encoded 768/1830
Encoded 800/1830
Encoded 832/1830
Encoded 864/1830
Encoded 896/1830
Encoded 928/1830
Encoded 960/1830
Encoded 992/1830
Encoded 1024/1830
Encoded 1056/1830
Encoded 1088/1830
Encoded 1120/1830
Encoded 1152/1830
Encoded 1184/1830
Encoded 1216/1830
Encoded 1248/1830
Encoded 1280/1830
Encoded 1312/1830
Encoded 1344/1830
Encoded 1376/1830
Encoded 1408/1830
Encoded 1440/1830
Encoded 1472/1830
Encoded 1504/1830
Encoded 1536/1830
Encoded 1568/1830
Encoded 1600/1830
Encoded 1

In [4]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import json
import numpy as np

E5_MODEL_NAME = "intfloat/e5-base-v2"
TOP_K = 3

# ============================================================
# Same anchor as previous SPARQLCE example
# ============================================================

example = eval_items[EXAMPLE_INDEX]
anchor_query = example["anchor_query"]
positives = example["positives"]

print("Using same anchor from EXAMPLE_INDEX:", EXAMPLE_INDEX)
print("\nANCHOR QUERY:")
print(anchor_query)


# ============================================================
# Helper: retrieve top-k from embedding map
# ============================================================

def retrieve_topk_from_emb_map(
    emb_map,
    anchor_query,
    global_corpus,
    positives,
    top_k=3,
):
    corpus_embs = np.vstack([emb_map[q] for q in global_corpus])
    anchor_emb = emb_map[anchor_query]

    scores = corpus_embs @ anchor_emb

    anchor_idx = global_corpus.index(anchor_query)
    scores = scores.copy()
    scores[anchor_idx] = np.finfo(scores.dtype).min

    order = np.argsort(-scores)

    top_items = []

    for rank, idx in enumerate(order[:top_k], start=1):
        q = global_corpus[idx]

        top_items.append({
            "rank": rank,
            "similarity_score": float(scores[idx]),
            "is_annotated_positive": q in positives,
            "query": q,
            "valid_perturbations": make_perturbations(q),
        })

    return top_items


# ============================================================
# SPARQLCE top-3
# If emb_map already exists from previous cell, reuse it.
# Otherwise rebuild it.
# ============================================================

if "emb_map" not in globals():
    print("\nSPARQLCE emb_map not found, rebuilding...")

    model = SPARQLEmbedder(str(CHECKPOINT_PATH))
    emb_map = {}

    for i in range(0, len(global_corpus), BATCH_SIZE):
        batch = global_corpus[i:i + BATCH_SIZE]
        embs = model.encode(batch)

        if not isinstance(embs, np.ndarray):
            embs = np.array(embs)

        embs = embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-12)

        for q, e in zip(batch, embs):
            emb_map[q] = e

        print(f"Encoded SPARQLCE {i + len(batch)}/{len(global_corpus)}")

sparqlce_top3 = retrieve_topk_from_emb_map(
    emb_map=emb_map,
    anchor_query=anchor_query,
    global_corpus=global_corpus,
    positives=positives,
    top_k=TOP_K,
)


# ============================================================
# E5 top-3
# ============================================================

print("\nLoading E5 model:", E5_MODEL_NAME)
e5_model = SentenceTransformer(E5_MODEL_NAME)

e5_emb_map = {}

for i in range(0, len(global_corpus), BATCH_SIZE):
    batch = global_corpus[i:i + BATCH_SIZE]

    # E5 expects query prefix for query-style retrieval
    batch_inputs = ["query: " + q for q in batch]

    embs = e5_model.encode(
        batch_inputs,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    for q, e in zip(batch, embs):
        e5_emb_map[q] = e

    print(f"Encoded E5 {i + len(batch)}/{len(global_corpus)}")

e5_top3 = retrieve_topk_from_emb_map(
    emb_map=e5_emb_map,
    anchor_query=anchor_query,
    global_corpus=global_corpus,
    positives=positives,
    top_k=TOP_K,
)


# ============================================================
# Final comparison JSON
# ============================================================

comparison_output = {
    "example_index": EXAMPLE_INDEX,
    "anchor": {
        "query": anchor_query,
        "valid_perturbations": make_perturbations(anchor_query),
    },
    "models": {
        "SPARQLCE": {
            "top_k": TOP_K,
            "retrieved_positive_count": sum(
                1 for x in sparqlce_top3 if x["is_annotated_positive"]
            ),
            "retrieved": sparqlce_top3,
        },
        "E5_base_v2": {
            "model_name": E5_MODEL_NAME,
            "top_k": TOP_K,
            "retrieved_positive_count": sum(
                1 for x in e5_top3 if x["is_annotated_positive"]
            ),
            "retrieved": e5_top3,
        },
    },
}

print("\n" + "=" * 100)
print("SPARQLCE vs E5 RETRIEVAL-STYLE ROBUSTNESS EXAMPLE")
print("=" * 100)

print(json.dumps(comparison_output, indent=2, ensure_ascii=False))


# ============================================================
# Optional compact table view
# ============================================================

rows = []

for model_name, top3 in [
    ("SPARQLCE", sparqlce_top3),
    ("E5_base_v2", e5_top3),
]:
    for item in top3:
        rows.append({
            "Model": model_name,
            "Rank": item["rank"],
            "Similarity": item["similarity_score"],
            "Annotated Positive": item["is_annotated_positive"],
            "Perturbations": len(item["valid_perturbations"]),
            "Query Preview": item["query"][:120].replace("\n", " ") + "...",
        })

comparison_df = pd.DataFrame(rows)
display(comparison_df)


# ============================================================
# Save JSON
# ============================================================

OUTPUT_PATH = PROJECT_ROOT / "dataset" / f"rq2_compare_sparqlce_e5_top3_example_{EXAMPLE_INDEX}.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(comparison_output, f, indent=2, ensure_ascii=False)

print("\nSaved comparison JSON to:")
print(OUTPUT_PATH)

Using same anchor from EXAMPLE_INDEX: 0

ANCHOR QUERY:
PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  dbpo: <http://dbpedia.org/ontology/>

SELECT DISTINCT  ?type
WHERE
  { dbpo:Food  rdf:type  ?type }

Loading E5 model: intfloat/e5-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1042.26it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoded E5 32/1830
Encoded E5 64/1830
Encoded E5 96/1830
Encoded E5 128/1830
Encoded E5 160/1830
Encoded E5 192/1830
Encoded E5 224/1830
Encoded E5 256/1830
Encoded E5 288/1830
Encoded E5 320/1830
Encoded E5 352/1830
Encoded E5 384/1830
Encoded E5 416/1830
Encoded E5 448/1830
Encoded E5 480/1830
Encoded E5 512/1830
Encoded E5 544/1830
Encoded E5 576/1830
Encoded E5 608/1830
Encoded E5 640/1830
Encoded E5 672/1830
Encoded E5 704/1830
Encoded E5 736/1830
Encoded E5 768/1830
Encoded E5 800/1830
Encoded E5 832/1830
Encoded E5 864/1830
Encoded E5 896/1830
Encoded E5 928/1830
Encoded E5 960/1830
Encoded E5 992/1830
Encoded E5 1024/1830
Encoded E5 1056/1830
Encoded E5 1088/1830
Encoded E5 1120/1830
Encoded E5 1152/1830
Encoded E5 1184/1830
Encoded E5 1216/1830
Encoded E5 1248/1830
Encoded E5 1280/1830
Encoded E5 1312/1830
Encoded E5 1344/1830
Encoded E5 1376/1830
Encoded E5 1408/1830
Encoded E5 1440/1830
Encoded E5 1472/1830
Encoded E5 1504/1830
Encoded E5 1536/1830
Encoded E5 1568/1830
Encod

,Model,Rank,Similarity,Annotated Positive,Perturbations,Query Preview
0,SPARQLCE,1,0.995021,True,4,PREFIX rdf: <http://www.w3.org/1999/02/22-rd...
1,SPARQLCE,2,0.994279,True,4,PREFIX rdf: <http://www.w3.org/1999/02/22-rd...
2,SPARQLCE,3,0.993934,True,4,PREFIX rdf: <http://www.w3.org/1999/02/22-rd...
3,E5_base_v2,1,0.984489,True,4,PREFIX rdf: <http://www.w3.org/1999/02/22-rd...
4,E5_base_v2,2,0.983966,False,4,PREFIX rdfs: <http://www.w3.org/2000/01/rdf-s...
5,E5_base_v2,3,0.978882,True,4,PREFIX rdf: <http://www.w3.org/1999/02/22-rd...



Saved comparison JSON to:
/local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/dataset/rq2_compare_sparqlce_e5_top3_example_0.json


In [8]:
# ============================================================
# One-example robustness case study:
# Same anchor, model-specific top-3 positives, then perturb each candidate
# Models: E5 + SPARQLCE
# ============================================================

import os
import sys
import re
import json
import random
from pathlib import Path
from typing import Any, Dict, List, Set, Tuple

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


# ============================================================
# Config
# ============================================================

SEED = 42
random.seed(SEED)

E5_MODEL_NAME = "intfloat/e5-base-v2"

# Choose one:
#   "top3_annotated_positives" = rank only annotated positives and take top 3
#   "top3_retrieved_any"       = take top 3 retrieved from full corpus, may include negatives
CANDIDATE_MODE = "top3_annotated_positives"

TOP_K = 3

# If None, the code selects the first anchor with at least 3 positives.
EXAMPLE_INDEX = None


# ============================================================
# Find project paths
# ============================================================

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "src" / "embedder.py").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

candidate_data_paths = [
    Path.cwd() / "dataset" / "hard_triplets_tfidf_dbpedia.json",
    Path.cwd().parent / "dataset" / "hard_triplets_tfidf_dbpedia.json",
    PROJECT_ROOT / "evaluation" / "dataset" / "hard_triplets_tfidf_dbpedia.json",
    PROJECT_ROOT / "dataset" / "hard_triplets_tfidf_dbpedia.json",
]

DATA_PATH = next((p for p in candidate_data_paths if p.exists()), candidate_data_paths[0])

candidate_checkpoint_paths = [
    PROJECT_ROOT / "checkpoints" / "dbpedia_model_semantic-20.pt",
    PROJECT_ROOT.parent / "checkpoints" / "dbpedia_model_semantic-20.pt",
    PROJECT_ROOT / "evaluation" / ".." / "checkpoints" / "dbpedia_model_semantic-20.pt",
]

CHECKPOINT_PATH = next((p for p in candidate_checkpoint_paths if p.exists()), candidate_checkpoint_paths[0])

print("DATA_PATH:", DATA_PATH)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)


# ============================================================
# Import SPARQLCE
# ============================================================

from src.embedder import SPARQLEmbedder


# ============================================================
# Retrieval dataset helpers
# ============================================================

EvalItem = Tuple[str, List[str], Set[str]]


def get_query_text(x: Any) -> str:
    if x is None:
        return ""

    if isinstance(x, dict):
        return str(
            x.get("query")
            or x.get("sparql")
            or x.get("sparql_query")
            or x.get("anchor_query")
            or ""
        ).strip()

    return str(x).strip()


def get_anchor_query(item: Dict[str, Any]) -> str:
    return get_query_text(
        item.get("anchor_query")
        or item.get("base_query")
        or item.get("query")
        or item.get("sparql")
    )


def iter_labeled_queries(item: Dict[str, Any], positives_only: bool = False) -> List[str]:
    out = []

    for p in item.get("positive_queries") or []:
        q = get_query_text(p)
        if q:
            out.append(q)

    if not positives_only:
        for n in item.get("negative_queries") or []:
            q = get_query_text(n)
            if q:
                out.append(q)

    return out


def load_triplets(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def build_global_corpus(raw_data: List[Dict[str, Any]]):
    unique_queries = []
    seen_q = set()

    def add_unique(q: str):
        if q and q not in seen_q:
            seen_q.add(q)
            unique_queries.append(q)

    for item in raw_data:
        anchor_q = get_anchor_query(item)
        if not anchor_q:
            continue

        add_unique(anchor_q)

        for q in iter_labeled_queries(item, positives_only=False):
            add_unique(q)

    return unique_queries, seen_q


def build_eval_items(
    raw_data: List[Dict[str, Any]],
    global_corpus: List[str],
    seen_q: Set[str],
) -> List[EvalItem]:

    eval_items = []

    for item in raw_data:
        anchor_q = get_anchor_query(item)

        if not anchor_q or anchor_q not in seen_q:
            continue

        pool = [q for q in global_corpus if q != anchor_q]

        positives = set()
        for p in item.get("positive_queries") or []:
            q = get_query_text(p)
            if q and q in seen_q and q != anchor_q:
                positives.add(q)

        if positives:
            eval_items.append((anchor_q, pool, positives))

    return eval_items


# ============================================================
# Perturbation helpers
# ============================================================

PREFIX_RE = re.compile(
    r"^\s*(?:PREFIX\s+[^\s:]*:?\s*<[^>]*>|BASE\s+<[^>]*>)\s*",
    re.IGNORECASE | re.DOTALL,
)

COUNT_RE = re.compile(r"\bCOUNT\s*\(", re.IGNORECASE)

VAR_TOKEN_RE = re.compile(
    r"(?<![A-Za-z0-9_])([?$][A-Za-z_][A-Za-z0-9_]*)(?![A-Za-z0-9_])"
)


def normalize_query(q: str) -> str:
    return " ".join(str(q or "").strip().split())


def remove_sparql_comments(q: str) -> str:
    q = str(q or "")

    out = []
    in_string = False
    string_char = None
    in_iri = False

    i = 0
    while i < len(q):
        ch = q[i]

        if in_string:
            out.append(ch)

            if ch == "\\":
                if i + 1 < len(q):
                    out.append(q[i + 1])
                    i += 2
                    continue

            if ch == string_char:
                in_string = False
                string_char = None

            i += 1
            continue

        if in_iri:
            out.append(ch)

            if ch == ">":
                in_iri = False

            i += 1
            continue

        if ch in ["'", '"']:
            in_string = True
            string_char = ch
            out.append(ch)
            i += 1
            continue

        if ch == "<":
            in_iri = True
            out.append(ch)
            i += 1
            continue

        if ch == "#":
            while i < len(q) and q[i] not in ["\n", "\r"]:
                i += 1
            out.append(" ")
            continue

        out.append(ch)
        i += 1

    return "".join(out)


def strip_prefixes_and_comments(q: str) -> str:
    q = remove_sparql_comments(q).strip()

    while True:
        m = PREFIX_RE.match(q)
        if not m:
            break
        q = q[m.end():].strip()

    return q


def has_select_star(q: str) -> bool:
    body = strip_prefixes_and_comments(q)
    return bool(re.search(r"(?is)^SELECT\s+(?:DISTINCT\s+|REDUCED\s+)?\*", body))


def find_where_body_bounds(query: str):
    m = re.search(r"\bWHERE\s*\{", query, re.IGNORECASE)

    if m:
        open_brace = query.find("{", m.start())
    else:
        open_brace = query.find("{")

    if open_brace < 0:
        return None

    depth = 0
    in_string = False
    string_char = None
    in_iri = False

    i = open_brace
    while i < len(query):
        ch = query[i]

        if in_string:
            if ch == "\\":
                i += 2
                continue

            if ch == string_char:
                in_string = False
                string_char = None

            i += 1
            continue

        if in_iri:
            if ch == ">":
                in_iri = False
            i += 1
            continue

        if ch in ["'", '"']:
            in_string = True
            string_char = ch
            i += 1
            continue

        if ch == "<":
            in_iri = True
            i += 1
            continue

        if ch == "{":
            depth += 1

        elif ch == "}":
            depth -= 1
            if depth == 0:
                return open_brace + 1, i

        i += 1

    return None


def collect_variables_outside_strings_and_iris(query: str):
    seen = []

    in_string = False
    string_char = None
    in_iri = False

    i = 0
    while i < len(query):
        ch = query[i]

        if in_string:
            if ch == "\\":
                i += 2
                continue

            if ch == string_char:
                in_string = False
                string_char = None

            i += 1
            continue

        if in_iri:
            if ch == ">":
                in_iri = False

            i += 1
            continue

        if ch in ["'", '"']:
            in_string = True
            string_char = ch
            i += 1
            continue

        if ch == "<":
            in_iri = True
            i += 1
            continue

        m = VAR_TOKEN_RE.match(query, i)
        if m:
            var_token = m.group(1)
            if var_token not in seen:
                seen.append(var_token)
            i = m.end()
            continue

        i += 1

    return seen


def replace_variables_outside_strings_and_iris(query: str, mapping: dict) -> str:
    out = []

    in_string = False
    string_char = None
    in_iri = False

    i = 0
    while i < len(query):
        ch = query[i]

        if in_string:
            out.append(ch)

            if ch == "\\":
                if i + 1 < len(query):
                    out.append(query[i + 1])
                    i += 2
                    continue

            if ch == string_char:
                in_string = False
                string_char = None

            i += 1
            continue

        if in_iri:
            out.append(ch)

            if ch == ">":
                in_iri = False

            i += 1
            continue

        if ch in ["'", '"']:
            in_string = True
            string_char = ch
            out.append(ch)
            i += 1
            continue

        if ch == "<":
            in_iri = True
            out.append(ch)
            i += 1
            continue

        m = VAR_TOKEN_RE.match(query, i)
        if m:
            old_var = m.group(1)
            out.append(mapping.get(old_var, old_var))
            i = m.end()
            continue

        out.append(ch)
        i += 1

    return "".join(out)


def rename_variables(query: str) -> str:
    """
    Readable variable renaming independent of original variable names.
    No numbers. No original variable name reuse.
    """
    seen = collect_variables_outside_strings_and_iris(query)

    if not seen:
        return query

    readable_pool = [
        "entity",
        "subject",
        "object",
        "resource",
        "result",
        "value",
        "label",
        "name",
        "type",
        "category",
        "classType",
        "property",
        "relation",
        "target",
        "source",
        "item",
        "node",
        "concept",
        "answer",
        "candidate",
    ]

    fallback_pool = [
        "alpha",
        "beta",
        "gamma",
        "delta",
        "epsilon",
        "zeta",
        "theta",
        "lambda",
        "omega",
    ]

    existing_vars = set(collect_variables_outside_strings_and_iris(query))
    used_new_vars = set()
    mapping = {}

    def next_available_name(base_name: str) -> str:
        candidate = f"?{base_name}"
        if candidate not in existing_vars and candidate not in used_new_vars:
            return candidate

        for suffix in fallback_pool:
            candidate2 = f"?{base_name}{suffix.capitalize()}"
            if candidate2 not in existing_vars and candidate2 not in used_new_vars:
                return candidate2

        candidate3 = f"?{base_name}Alt"
        while candidate3 in existing_vars or candidate3 in used_new_vars:
            candidate3 += "Alt"

        return candidate3

    for i, old_var in enumerate(seen):
        base = readable_pool[i] if i < len(readable_pool) else "entity"
        new_var = next_available_name(base)
        mapping[old_var] = new_var
        used_new_vars.add(new_var)

    return replace_variables_outside_strings_and_iris(query, mapping)


def split_top_level_dot_patterns(where_body: str):
    parts = []
    buf = []

    depth = 0
    in_string = False
    string_char = None
    in_iri = False

    i = 0
    while i < len(where_body):
        ch = where_body[i]

        if in_string:
            buf.append(ch)

            if ch == "\\":
                if i + 1 < len(where_body):
                    buf.append(where_body[i + 1])
                    i += 2
                    continue

            if ch == string_char:
                in_string = False
                string_char = None

            i += 1
            continue

        if in_iri:
            buf.append(ch)

            if ch == ">":
                in_iri = False

            i += 1
            continue

        if ch in ["'", '"']:
            in_string = True
            string_char = ch
            buf.append(ch)
            i += 1
            continue

        if ch == "<":
            in_iri = True
            buf.append(ch)
            i += 1
            continue

        if ch == "{":
            depth += 1
            buf.append(ch)
            i += 1
            continue

        if ch == "}":
            depth -= 1
            buf.append(ch)
            i += 1
            continue

        if ch == "." and depth == 0:
            part = "".join(buf).strip()
            if part:
                parts.append(part)

            buf = []
            i += 1
            continue

        buf.append(ch)
        i += 1

    tail = "".join(buf).strip()

    if tail:
        parts.append(tail)

    return parts


def is_safe_reorder_pattern(pattern: str) -> bool:
    p = pattern.strip()
    upper = p.upper()

    unsafe_keywords = [
        "BIND",
        "OPTIONAL",
        "UNION",
        "VALUES",
        "SERVICE",
        "GRAPH",
        "MINUS",
        "SELECT",
        "ASK",
        "CONSTRUCT",
        "DESCRIBE",
    ]

    for keyword in unsafe_keywords:
        if re.search(rf"\b{keyword}\b", upper):
            return False

    if re.search(r"\bFILTER\s+EXISTS\b", upper):
        return False

    if re.search(r"\bFILTER\s+NOT\s+EXISTS\b", upper):
        return False

    return True


def reorder_triples(query: str) -> str:
    bounds = find_where_body_bounds(query)

    if not bounds:
        return query

    start, end = bounds
    body = query[start:end]

    patterns = split_top_level_dot_patterns(body)

    if len(patterns) <= 1:
        return query

    if not all(is_safe_reorder_pattern(p) for p in patterns):
        return query

    shuffled = patterns[:]
    random.shuffle(shuffled)

    if shuffled == patterns:
        shuffled = shuffled[::-1]

    new_body = "\n  " + " .\n  ".join(shuffled) + " .\n"

    return query[:start] + new_body + query[end:]


def insert_dummy_optional(query: str) -> str:
    if has_select_star(query):
        return query

    bounds = find_where_body_bounds(query)

    if not bounds:
        return query

    _, end = bounds

    noop_optional = """
  OPTIONAL {
    FILTER(false)
  }
"""

    return query[:end] + noop_optional + query[end:]


def inject_limit_offset(query: str) -> str:
    if re.search(r"\bOFFSET\b", query, re.IGNORECASE):
        return query

    stripped = query.rstrip().rstrip(";").rstrip()

    return stripped + "\nOFFSET 0"


def combined_perturbation(query: str) -> str:
    q = rename_variables(query)
    q = reorder_triples(q)
    q = insert_dummy_optional(q)
    q = inject_limit_offset(q)
    return q


PERTURBATION_FUNCS = {
    "variable_renaming": rename_variables,
    "triple_reordering": reorder_triples,
    "optional_insertion": insert_dummy_optional,
    "limit_offset_injection": inject_limit_offset,
    "combined": combined_perturbation,
}


def make_perturbations(query: str) -> List[Dict[str, Any]]:
    perturbations = []

    for name, func in PERTURBATION_FUNCS.items():
        try:
            perturbed = func(query)
        except Exception as e:
            perturbations.append({
                "type": name,
                "query": query,
                "changed": False,
                "error": str(e),
            })
            continue

        perturbations.append({
            "type": name,
            "query": perturbed,
            "changed": normalize_query(perturbed) != normalize_query(query),
            "expected_answer_jaccard": 1.0,
            "validated_with_endpoint": False,
        })

    return perturbations


# ============================================================
# Embedding helpers
# ============================================================

def l2_normalize(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    a = l2_normalize(np.asarray(a))
    b = l2_normalize(np.asarray(b))
    return float(np.dot(a, b))


def encode_e5(model: SentenceTransformer, queries: List[str], batch_size: int = 32) -> np.ndarray:
    texts = ["query: " + q for q in queries]
    embs = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return np.asarray(embs)


def encode_sparqlce(model: SPARQLEmbedder, queries: List[str], batch_size: int = 32) -> np.ndarray:
    all_embs = []

    for i in range(0, len(queries), batch_size):
        batch = queries[i:i + batch_size]
        embs = model.encode(batch)
        embs = np.asarray(embs)
        embs = l2_normalize(embs)
        all_embs.append(embs)

    return np.vstack(all_embs)


def build_emb_map(queries: List[str], embs: np.ndarray) -> Dict[str, np.ndarray]:
    return {q: e for q, e in zip(queries, embs)}


def rank_candidates(
    anchor_q: str,
    candidates: List[str],
    emb_map: Dict[str, np.ndarray],
    top_k: int,
) -> List[Tuple[str, float]]:
    anchor_emb = emb_map[anchor_q]

    scored = []
    for q in candidates:
        if q == anchor_q:
            continue
        scored.append((q, cosine(anchor_emb, emb_map[q])))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


# ============================================================
# Load data
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(f"DATA_PATH not found: {DATA_PATH}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"CHECKPOINT_PATH not found: {CHECKPOINT_PATH}")

raw_data = load_triplets(str(DATA_PATH))
global_corpus, seen_q = build_global_corpus(raw_data)
eval_items = build_eval_items(raw_data, global_corpus, seen_q)

print(f"Loaded raw items: {len(raw_data)}")
print(f"Global corpus size: {len(global_corpus)}")
print(f"Eval anchors with positives: {len(eval_items)}")

if not eval_items:
    raise ValueError("No eval items with positives found.")


# ============================================================
# Pick one example anchor
# ============================================================

if EXAMPLE_INDEX is None:
    chosen_idx = None
    for i, item in enumerate(eval_items):
        _, _, positives = item
        if len(positives) >= TOP_K:
            chosen_idx = i
            break

    if chosen_idx is None:
        chosen_idx = 0
else:
    chosen_idx = EXAMPLE_INDEX

anchor_q, pool, positives = eval_items[chosen_idx]

print("\n" + "=" * 100)
print(f"Chosen eval item index: {chosen_idx}")
print(f"Number of annotated positives for this anchor: {len(positives)}")
print("=" * 100)
print(anchor_q)


# ============================================================
# Load models and encode global corpus
# ============================================================

print("\nLoading E5...")
e5_model = SentenceTransformer(E5_MODEL_NAME)

print("Encoding global corpus with E5...")
e5_global_embs = encode_e5(e5_model, global_corpus)
e5_emb_map = build_emb_map(global_corpus, e5_global_embs)

print("\nLoading SPARQLCE...")
sparqlce_model = SPARQLEmbedder(str(CHECKPOINT_PATH))

print("Encoding global corpus with SPARQLCE...")
sparqlce_global_embs = encode_sparqlce(sparqlce_model, global_corpus)
sparqlce_emb_map = build_emb_map(global_corpus, sparqlce_global_embs)


# ============================================================
# Select model-specific top-3 candidates
# ============================================================

if CANDIDATE_MODE == "top3_annotated_positives":
    candidate_pool = list(positives)
elif CANDIDATE_MODE == "top3_retrieved_any":
    candidate_pool = [q for q in global_corpus if q != anchor_q]
else:
    raise ValueError("CANDIDATE_MODE must be 'top3_annotated_positives' or 'top3_retrieved_any'.")

model_configs = {
    "E5": {
        "model": e5_model,
        "emb_map": e5_emb_map,
        "encode_func": lambda qs: encode_e5(e5_model, qs),
    },
    "SPARQLCE": {
        "model": sparqlce_model,
        "emb_map": sparqlce_emb_map,
        "encode_func": lambda qs: encode_sparqlce(sparqlce_model, qs),
    },
}

rows = []
example_json = {
    "anchor_query": anchor_q,
    "candidate_mode": CANDIDATE_MODE,
    "top_k": TOP_K,
    "models": {},
}

for model_name, cfg in model_configs.items():
    emb_map = cfg["emb_map"]
    encode_func = cfg["encode_func"]

    ranked = rank_candidates(
        anchor_q=anchor_q,
        candidates=candidate_pool,
        emb_map=emb_map,
        top_k=TOP_K,
    )

    example_json["models"][model_name] = []

    anchor_emb = emb_map[anchor_q]

    for rank, (candidate_q, original_sim) in enumerate(ranked, start=1):
        perturbations = make_perturbations(candidate_q)
        perturbed_queries = [p["query"] for p in perturbations]

        perturbed_embs = encode_func(perturbed_queries)

        candidate_record = {
            "rank": rank,
            "is_annotated_positive": candidate_q in positives,
            "original_query": candidate_q,
            "original_similarity_to_anchor": original_sim,
            "perturbations": [],
        }

        for p, p_emb in zip(perturbations, perturbed_embs):
            pert_sim = cosine(anchor_emb, p_emb)
            drop = original_sim - pert_sim

            rows.append({
                "Model": model_name,
                "Rank": rank,
                "Is annotated positive": candidate_q in positives,
                "Perturbation": p["type"],
                "Original sim anchor-positive": round(original_sim, 6),
                "Perturbed sim anchor-perturbedPositive": round(pert_sim, 6),
                "Drop": round(drop, 6),
                "Changed": p["changed"],
            })

            candidate_record["perturbations"].append({
                "type": p["type"],
                "query": p["query"],
                "changed": p["changed"],
                "similarity_to_anchor": pert_sim,
                "drop_from_original": drop,
                "expected_answer_jaccard": p.get("expected_answer_jaccard", 1.0),
                "validated_with_endpoint": p.get("validated_with_endpoint", False),
            })

        example_json["models"][model_name].append(candidate_record)


# ============================================================
# Display results
# ============================================================

df = pd.DataFrame(rows)

print("\nCandidate perturbation robustness:")
print("Cosine(anchor, positive) vs cosine(anchor, perturbed positive)")
display(df)

SAVE_PATH = Path.cwd() / "one_example_top3_positive_perturbation_robustness.json"

with SAVE_PATH.open("w", encoding="utf-8") as f:
    json.dump(example_json, f, indent=2, ensure_ascii=False)

print(f"\nSaved example JSON to: {SAVE_PATH}")

print("\nPreview JSON keys:")
print(json.dumps({
    "anchor_query": example_json["anchor_query"][:200] + "...",
    "candidate_mode": example_json["candidate_mode"],
    "models": list(example_json["models"].keys()),
}, indent=2))

PROJECT_ROOT: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings
DATA_PATH: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/evaluation/dataset/hard_triplets_tfidf_dbpedia.json
CHECKPOINT_PATH: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/checkpoints/dbpedia_model_semantic-20.pt
Loaded raw items: 1428
Global corpus size: 1830
Eval anchors with positives: 1428

Chosen eval item index: 0
Number of annotated positives for this anchor: 31
PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  dbpo: <http://dbpedia.org/ontology/>

SELECT DISTINCT  ?type
WHERE
  { dbpo:Food  rdf:type  ?type }

Loading E5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1164.24it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding global corpus with E5...

Loading SPARQLCE...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1497.01it/s, Materializing param=pooler.dense.weight]                               


Loading model from /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/checkpoints/dbpedia_model_semantic-20.pt...
Encoding global corpus with SPARQLCE...

Candidate perturbation robustness:
Cosine(anchor, positive) vs cosine(anchor, perturbed positive)


,Model,Rank,Is annotated positive,Perturbation,Original sim anchor-positive,Perturbed sim anchor-perturbedPositive,Drop,Changed
0,E5,1,True,variable_renaming,0.984489,0.977625,0.006864,True
1,E5,1,True,triple_reordering,0.984489,0.984489,0.000000,False
2,E5,1,True,optional_insertion,0.984489,0.973700,0.010789,True
3,E5,1,True,limit_offset_injection,0.984489,0.970173,0.014316,True
4,E5,1,True,combined,0.984489,0.961317,0.023172,True
5,E5,2,True,variable_renaming,0.978882,0.971842,0.007040,True
6,E5,2,True,triple_reordering,0.978882,0.978882,0.000000,False
7,E5,2,True,optional_insertion,0.978882,0.965796,0.013087,True
8,E5,2,True,limit_offset_injection,0.978882,0.962343,0.016540,True
9,E5,2,True,combined,0.978882,0.952429,0.026454,True



Saved example JSON to: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/evaluation/one_example_top3_positive_perturbation_robustness.json

Preview JSON keys:
{
  "anchor_query": "PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\nPREFIX  dbpo: <http://dbpedia.org/ontology/>\n\nSELECT DISTINCT  ?type\nWHERE\n  { dbpo:Food  rdf:type  ?type }...",
  "candidate_mode": "top3_annotated_positives",
  "models": [
    "E5",
    "SPARQLCE"
  ]
}
